<p align="center"><img src="../docs/logo.jpg" alt="GIK-IceChain" width="480"/></p>

# Benchmark Report: GIK+IceChunk vs dynamical.org

**GIK-IceChain v2.0 — ECMWF Code for Earth 2026**

This notebook benchmarks two approaches to accessing the ECMWF IFS ensemble archive
for the East Africa domain:

| Approach | Description | Storage |
|----------|-------------|----------|
| **GIK+IceChunk** | Virtual store — byte-range references only, no data duplication | ~18.5 GB metadata |
| **dynamical.org** | Full-copy IceChunk Zarr store — complete data downloaded to S3 | ~242 TB |

Metrics measured:
- Time-to-first-byte (cold read)
- Full-scan elapsed time (30-day East Africa domain mean)
- Estimated S3 egress cost
- Dask scalability (elapsed time vs. number of workers)

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

RESULTS_DIR = REPO_ROOT / "results" / "benchmarks"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Run the benchmark (requires AWS credentials)

Skip this cell and load pre-computed results in section 2 if you do not have
access to the stores.

In [ ]:
RUN_LIVE = False  # set True to execute against live S3 stores

GIK_STORE_URI        = os.getenv("GIK_ICECHUNK_STORE_URI", "")
DYNAMICAL_STORE_URI  = os.getenv("DYNAMICAL_STORE_URI",   "")

if RUN_LIVE and GIK_STORE_URI and DYNAMICAL_STORE_URI:
    from gik_icechain.conversion.benchmark import run_benchmark

    results = run_benchmark(
        gik_store_uri=GIK_STORE_URI,
        dynamical_store_uri=DYNAMICAL_STORE_URI,
        domain="east_africa",
        n_days=30,
        n_workers=4,
        output_dir=str(RESULTS_DIR),
    )
    print("Benchmark complete. Results:", results)
else:
    print("Skipping live benchmark — loading pre-computed results.")

## 2. Load results

If a benchmark CSV exists, load it. Otherwise use the reference numbers from
the preliminary run documented in the README.

In [ ]:
csv_files = sorted(RESULTS_DIR.glob("benchmark_east_africa_*.csv"))

if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f"Loaded: {csv_files[-1].name}")
else:
    # Reference numbers from README — preliminary 30-day window
    df = pd.DataFrame([
        {
            "approach":             "GIK+IceChunk",
            "n_days":               30,
            "domain":               "east_africa",
            "store_size_gb":        18.5,
            "time_to_first_byte_s": 1.8,
            "full_scan_elapsed_s":  2700.0,  # ~45 min at 32 vCPU
            "data_read_gb":         0.22,
            "estimated_egress_usd": 0.02,
            "n_workers":            32,
        },
        {
            "approach":             "dynamical.org",
            "n_days":               30,
            "domain":               "east_africa",
            "store_size_gb":        242_000.0,
            "time_to_first_byte_s": 0.9,
            "full_scan_elapsed_s":  1200.0,  # ~20 min at 32 vCPU
            "data_read_gb":         0.22,
            "estimated_egress_usd": 0.02,
            "n_workers":            32,
        },
    ])
    print("Using reference numbers from README (no CSV found).")

df

## 3. Summary table

In [ ]:
summary = df[[
    "approach",
    "store_size_gb",
    "time_to_first_byte_s",
    "full_scan_elapsed_s",
    "estimated_egress_usd",
]].copy()

summary["store_size"]    = summary["store_size_gb"].apply(
    lambda x: f"{x:.1f} GB" if x < 1000 else f"{x/1000:.0f} TB"
)
summary["ttfb"]          = summary["time_to_first_byte_s"].apply(lambda x: f"{x:.1f} s")
summary["scan_30d"]      = summary["full_scan_elapsed_s"].apply(
    lambda x: f"{x/60:.0f} min" if x >= 60 else f"{x:.0f} s"
)
summary["egress_usd"]    = summary["estimated_egress_usd"].apply(lambda x: f"~${x:.2f}/day")

display_cols = ["approach", "store_size", "ttfb", "scan_30d", "egress_usd"]
print(summary[display_cols].to_markdown(index=False))

## 4. Storage comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

approaches = df["approach"].tolist()
colors = ["#1f77b4", "#ff7f0e"]

# Storage
ax = axes[0]
bars = ax.bar(approaches, df["store_size_gb"], color=colors)
ax.set_yscale("log")
ax.set_ylabel("Store size (GB, log scale)")
ax.set_title("Storage footprint")
for bar, val in zip(bars, df["store_size_gb"]):
    label = f"{val:.0f} GB" if val < 1000 else f"{val/1000:.0f} TB"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.1,
            label, ha="center", va="bottom", fontsize=9)

# Time-to-first-byte
ax = axes[1]
bars = ax.bar(approaches, df["time_to_first_byte_s"], color=colors)
ax.set_ylabel("Seconds")
ax.set_title("Time-to-first-byte")
for bar, val in zip(bars, df["time_to_first_byte_s"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.1f} s", ha="center", va="bottom", fontsize=9)

# Full-scan elapsed time
ax = axes[2]
bars = ax.bar(approaches, df["full_scan_elapsed_s"] / 60, color=colors)
ax.set_ylabel("Minutes")
ax.set_title("Full scan — 30 days (East Africa)")
for bar, val in zip(bars, df["full_scan_elapsed_s"] / 60):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{val:.0f} min", ha="center", va="bottom", fontsize=9)

fig.suptitle(
    f"GIK+IceChunk vs dynamical.org — East Africa, {df['n_days'].iloc[0]}-day window",
    fontsize=13, fontweight="bold",
)
fig.tight_layout()
out_path = RESULTS_DIR / "benchmark_comparison.png"
fig.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## 5. Compression ratio

The key metric: how much metadata does GIK+IceChunk require relative to the
raw GRIB2 archive size?

In [ ]:
RAW_GRIB2_TB = 1_000.0  # ~1 PB archive
gik_gb = df.loc[df["approach"] == "GIK+IceChunk", "store_size_gb"].values[0]

compression_ratio = (RAW_GRIB2_TB * 1000) / gik_gb
print(f"Raw GRIB2 archive:    ~{RAW_GRIB2_TB:.0f} TB")
print(f"GIK+IceChunk store:   {gik_gb:.1f} GB (metadata only)")
print(f"Compression ratio:    {compression_ratio:,.0f}×")

## 6. Dask scalability

Projected scan time for the GIK+IceChunk approach as a function of the number
of Dask workers, assuming near-linear scaling (S3 throughput-limited).

In [ ]:
gik_row = df[df["approach"] == "GIK+IceChunk"].iloc[0]
base_workers = int(gik_row["n_workers"])
base_elapsed = float(gik_row["full_scan_elapsed_s"])

worker_counts = np.array([1, 2, 4, 8, 16, 32, 64])
# S3 bandwidth saturates; model as sqrt scaling beyond 16 workers
scaling = np.where(
    worker_counts <= 16,
    base_elapsed * base_workers / worker_counts,
    base_elapsed * base_workers / 16 * np.sqrt(16 / worker_counts),
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(worker_counts, scaling / 60, "o-", color="#1f77b4", linewidth=2)
ax.axvline(base_workers, linestyle="--", color="gray", alpha=0.6, label=f"Reference ({base_workers} workers)")
ax.set_xlabel("Number of Dask workers")
ax.set_ylabel("Estimated scan time (minutes)")
ax.set_title("GIK+IceChunk — scalability projection (30-day scan)")
ax.set_xscale("log", base=2)
ax.set_xticks(worker_counts)
ax.set_xticklabels(worker_counts)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
out_path = RESULTS_DIR / "benchmark_scalability.png"
fig.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## 7. Key takeaways

| Metric | GIK+IceChunk | dynamical.org | Winner |
|--------|-------------|--------------|--------|
| Storage | **18.5 GB** metadata | ~242 TB full copy | GIK (×13 000 smaller) |
| Time-to-first-byte | ~2 s | < 1 s | dynamical.org |
| 30-day scan (32 vCPU) | ~45 min | ~20 min | dynamical.org |
| Egress cost | ~$0.02/day | ~$0.02/day | Tie |
| Maintenance cost | **$0** (references public S3) | High (242 TB replication) | GIK |

**Conclusion**: GIK+IceChunk is the only zero-cost approach for organisations
without budget to maintain a full data copy. The ~2× scan overhead is acceptable
for retrospective analysis workflows. For latency-sensitive operational use,
a hybrid approach (pre-stage hot data, virtual for the archive) is optimal.